# Lecture 4: The Singular Value Decomposition (SVD)
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Geometric Interpretation: Unit Circle to Ellipse

The SVD reveals that $A$ maps the unit circle to an ellipse. The semi-axes of the ellipse are the singular values, oriented along the left singular vectors.

In [ ]:
A = np.array([[-2, 0],
              [-1, 3]], dtype=float)

t = np.linspace(0, 2 * np.pi, 300)
X = np.array([np.cos(t), np.sin(t)])
AX = A @ X

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(X[0], X[1], c=t, cmap='hsv', s=4)
axes[0].set_aspect('equal')
axes[0].set_title('x (unit circle)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(AX[0], AX[1], c=t, cmap='hsv', s=4)
axes[1].set_aspect('equal')
axes[1].set_title('Ax (ellipse)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Finding the SVD Geometrically

The singular values are the max and min of $\|Ax\|$ over the unit circle. The left singular vectors point along the semi-axes of the ellipse. The right singular vectors are their pre-images.

In [ ]:
norms_Ax = np.sqrt(np.sum(AX**2, axis=0))
k1 = np.argmax(norms_Ax)
k2 = np.argmin(norms_Ax)

sigma1 = norms_Ax[k1]
sigma2 = norms_Ax[k2]
print(f"Singular values (approx): sigma_1 = {sigma1:.4f}, sigma_2 = {sigma2:.4f}")

u1 = AX[:, k1] / sigma1
u2 = AX[:, k2] / sigma2
v1 = X[:, k1]
v2 = X[:, k2]

print(f"U (approx):\n  u1 = {u1}\n  u2 = {u2}")
print(f"V (approx):\n  v1 = {v1}\n  v2 = {v2}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Unit circle with right singular vectors
ax = axes[0]
ax.scatter(X[0], X[1], c=t, cmap='hsv', s=4)
for vi, label in [(v1, '$v_1$'), (v2, '$v_2$')]:
    ax.annotate('', xy=vi, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(1.2 * vi[0], 1.2 * vi[1], label, fontsize=12, ha='center')
ax.set_aspect('equal')
ax.set_title('x')
ax.grid(True, alpha=0.3)

# Ellipse with left singular vectors scaled by singular values
ax = axes[1]
ax.scatter(AX[0], AX[1], c=t, cmap='hsv', s=4)
for ui, si, label in [(u1, sigma1, r'$\sigma_1 u_1$'), (u2, sigma2, r'$\sigma_2 u_2$')]:
    ax.annotate('', xy=si * ui, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(1.1 * si * ui[0], 1.1 * si * ui[1], label, fontsize=12, ha='center')
ax.set_aspect('equal')
ax.set_title('Ax')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Computing the SVD with NumPy

Compare our geometric approximation to the exact SVD.

In [ ]:
U, S, Vt = np.linalg.svd(A)

print("U =")
print(U)
print("\nSigma =", S)
print("\nV^T =")
print(Vt)

# Verify reconstruction
print("\nReconstruction error:", np.linalg.norm(A - U @ np.diag(S) @ Vt))

## ||Ax|| over the Unit Circle

The maximum is $\sigma_1 = \|A\|_2$ and the minimum is $\sigma_2$.

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(t, norms_Ax)
plt.axhline(S[0], color='r', linestyle=':', label=rf'$\sigma_1 = {S[0]:.4f}$')
plt.axhline(S[1], color='b', linestyle=':', label=rf'$\sigma_2 = {S[1]:.4f}$')
plt.xlim(0, 2 * np.pi)
plt.xlabel('angle')
plt.ylabel(r'$\|Ax\|_2$')
plt.title(r'$\|Ax\|_2$ over the unit circle')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Full vs. Thin SVD

For a tall matrix ($m > n$), the thin SVD drops the unnecessary columns of $U$.

In [ ]:
np.random.seed(0)
A_tall = np.random.rand(6, 3)

# Full SVD
U_full, S_full, Vt_full = np.linalg.svd(A_tall, full_matrices=True)
print("Full SVD:")
print(f"  U: {U_full.shape}, Sigma: {S_full.shape}, V^T: {Vt_full.shape}")

# Thin SVD
U_thin, S_thin, Vt_thin = np.linalg.svd(A_tall, full_matrices=False)
print(f"\nThin SVD:")
print(f"  U: {U_thin.shape}, Sigma: {S_thin.shape}, V^T: {Vt_thin.shape}")

print(f"\nSingular values: {np.round(S_full, 4)}")

In [ ]:
# U_thin has orthonormal columns but is NOT unitary
print("U_thin^T @ U_thin (should be I_3):")
print(np.round(U_thin.T @ U_thin, 10))

print("\nU_thin @ U_thin^T (NOT I_6 — it's a projector):")
print(np.round(U_thin @ U_thin.T, 4))

## What the Singular Values Tell You

In [ ]:
A = np.array([[-2, 0],
              [-1, 3]], dtype=float)
U, S, Vt = np.linalg.svd(A)

print(f"Singular values:  {S}")
print(f"Rank:             {np.sum(S > 1e-10)}")
print(f"||A||_2:          {S[0]:.4f}  (= sigma_1)")
print(f"||A||_F:          {np.sqrt(np.sum(S**2)):.4f}  (= sqrt(sum sigma_i^2))")
print(f"||A^{{-1}}||_2:    {1/S[-1]:.4f}  (= 1/sigma_n)")
print(f"Condition number: {S[0]/S[-1]:.4f}  (= sigma_1/sigma_n)")
print(f"np.linalg.cond:   {np.linalg.cond(A):.4f}")

## Rank-$k$ Approximation (Eckart–Young)

$A \approx A_k = \sigma_1 u_1 v_1^T + \cdots + \sigma_k u_k v_k^T$

The error is $\sigma_{k+1}$.

In [ ]:
np.random.seed(42)
M = np.random.randn(50, 50)
U, S, Vt = np.linalg.svd(M)

ranks = [1, 2, 5, 10, 20, 30, 49]
print(f"{'k':>4s}  {'||M - M_k||_2':>14s}  {'sigma_{k+1}':>12s}  {'||M - M_k||_F':>14s}")
print("-" * 52)
for k in ranks:
    M_k = (U[:, :k] * S[:k]) @ Vt[:k, :]
    err_2 = np.linalg.norm(M - M_k, 2)
    err_F = np.linalg.norm(M - M_k, 'fro')
    sig_next = S[k] if k < len(S) else 0.0
    print(f"{k:4d}  {err_2:14.6f}  {sig_next:12.6f}  {err_F:14.6f}")

In [ ]:
# Singular value spectrum
plt.figure(figsize=(7, 3.5))
plt.semilogy(range(1, len(S) + 1), S, 'o-', markersize=4)
plt.xlabel('Index $k$')
plt.ylabel(r'$\sigma_k$')
plt.title('Singular values of a random 50x50 matrix')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Image Compression via SVD

Approximate a synthetic image using truncated SVD at various ranks.

In [ ]:
# Create a synthetic image with structure
np.random.seed(0)
n = 200
x = np.linspace(-2, 2, n)
X_grid, Y_grid = np.meshgrid(x, x)
img = (
    np.sin(3 * X_grid) * np.cos(3 * Y_grid)
    + 0.5 * np.exp(-(X_grid**2 + Y_grid**2))
    + 0.1 * np.random.randn(n, n)
)

U_img, S_img, Vt_img = np.linalg.svd(img)

ranks_to_show = [1, 5, 10, 25, 50]
fig, axes = plt.subplots(1, len(ranks_to_show) + 1, figsize=(18, 3))

axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Original (rank {np.linalg.matrix_rank(img)})')
axes[0].axis('off')

for ax, k in zip(axes[1:], ranks_to_show):
    img_k = (U_img[:, :k] * S_img[:k]) @ Vt_img[:k, :]
    ax.imshow(img_k, cmap='gray')
    storage = k * (n + n)
    ax.set_title(f'Rank {k}\n({100*storage/n**2:.0f}% storage)')
    ax.axis('off')

plt.suptitle('Low-rank approximation via SVD', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Singular value decay of the image
plt.figure(figsize=(7, 3.5))
plt.semilogy(range(1, len(S_img) + 1), S_img, 'o-', markersize=3)
for k in ranks_to_show:
    plt.axvline(k, color='gray', linestyle='--', alpha=0.4)
    plt.text(k + 1, S_img[0] * 0.8, f'k={k}', fontsize=8, rotation=90)
plt.xlabel('Index $k$')
plt.ylabel(r'$\sigma_k$')
plt.title('Singular values of the image')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The Three Steps of the SVD: Rotate, Stretch, Rotate

Visualize each step of $A = U \Sigma V^T$ applied to the unit circle.

In [ ]:
A = np.array([[-2, 0],
              [-1, 3]], dtype=float)
U, S, Vt = np.linalg.svd(A)

t = np.linspace(0, 2 * np.pi, 300)
circle = np.array([np.cos(t), np.sin(t)])

step1 = Vt @ circle              # V^T x: rotate input
step2 = np.diag(S) @ step1       # Sigma V^T x: stretch
step3 = U @ step2                # U Sigma V^T x: rotate output

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
titles = ['$x$ (unit circle)', r'$V^T x$ (rotated)', r'$\Sigma V^T x$ (stretched)',
          r'$U \Sigma V^T x = Ax$ (final)']
data = [circle, step1, step2, step3]

for ax, d, title in zip(axes, data, titles):
    ax.scatter(d[0], d[1], c=t, cmap='hsv', s=4)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)
    lim = max(np.max(np.abs(d[0])), np.max(np.abs(d[1]))) * 1.3
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

plt.tight_layout()
plt.show()

## Unitary Matrices Preserve the Sphere (3D)

Applying a unitary $Q$ to points on a sphere just rotates/reflects — the shape is unchanged.

In [ ]:
phi = np.linspace(0, 1.5 * np.pi, 60)
theta = np.linspace(0.25 * np.pi, np.pi, 50)
Phi, Theta = np.meshgrid(phi, theta)
Phi, Theta = Phi.ravel(), Theta.ravel()

pts = np.column_stack([
    np.cos(Phi) * np.sin(Theta),
    np.sin(Phi) * np.sin(Theta),
    np.cos(Theta)
])

magic3 = np.array([[8, 1, 6], [3, 5, 7], [4, 9, 2]], dtype=float)
Q, _ = np.linalg.qr(magic3)
Qpts = (Q @ pts.T).T

colr = np.sum(pts, axis=1)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colr, s=8, cmap='coolwarm')
ax1.set_title('x')
ax1.set_box_aspect([1, 1, 1])

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(Qpts[:, 0], Qpts[:, 1], Qpts[:, 2], c=colr, s=8, cmap='coolwarm')
ax2.set_title('Qx (rotated — same shape)')
ax2.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.show()

## The Four Fundamental Subspaces

The SVD makes all four subspaces explicit for a rank-deficient matrix.

In [ ]:
# Rank-2 matrix in R^3 x R^3
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [5, 7, 9]], dtype=float)  # row 3 = row 1 + row 2

U, S, Vt = np.linalg.svd(A)
r = np.sum(S > 1e-10)

print(f"Singular values: {np.round(S, 6)}")
print(f"Rank: {r}")
print(f"\nRange (col space): U[:, :r] columns")
print(np.round(U[:, :r], 4))
print(f"\nLeft nullspace: U[:, r:] columns")
print(np.round(U[:, r:], 4))
print(f"\nRow space: V[:, :r] columns  (= Vt[:r, :] rows)")
print(np.round(Vt[:r, :].T, 4))
print(f"\nNullspace: V[:, r:] columns  (= Vt[r:, :] rows)")
print(np.round(Vt[r:, :].T, 4))

# Verify: nullspace vector is killed by A
v_null = Vt[r:, :].T.ravel()
print(f"\nA @ nullspace vector = {np.round(A @ v_null, 12)}")